# 04 — Phase 4: Feature Engineering (merged)

**Purpose.** Runs the merged `sql/005_phase4_features.sql`, which builds `quarterly_modeling_table` (one row per county-quarter: the decline label plus six features measured one quarter earlier), and then `sql/003b_cv_fold_membership.sql`, which rebuilds the `cv_fold_rows` view that reads it. The notebook checks the table, compares it with Jolenn's pandas feature matrix, and shows how the two test-set conventions differ.

**The six features** (Jolenn's column names, used by her model outputs)

| Column | Meaning |
|---|---|
| `momentum_1m`, `momentum_3m`, `momentum_6m` | Relative change in the county's ZHVI over 1, 3 and 6 months, measured at the end of the quarter before the label quarter |
| `mortgage_rate_level` | National 30-year rate at that quarter end |
| `mortgage_rate_roc` | 12-month change in that rate, in **percentage points** |
| `rate_roc_x_momentum` | `mortgage_rate_roc` x `momentum_3m` |

**Decisions carried into this merged phase**

| Decision | Choice | Why |
|---|---|---|
| Rate-of-change | Percentage points, `rate(t) - rate(t-12)` | Same as Phase 3 and Jolenn's features |
| Interaction term | Rate change x **3-month** momentum | Jolenn's choice; 3 months is one quarter, the prediction horizon. The earlier SQL used 1-month |
| Alignment | Features from the preceding quarter | Predict one quarter ahead |
| Missing values | Drop a row if ANY of the six features is missing | The earlier SQL only checked two of them |
| Row dating | By **label** quarter (the convention behind the Phase 0 counts) | Jolenn's matrix is dated by feature quarter; section 7 shows the effect |

**Prerequisites.** Phase 2's `003_phase2_split.sql` has run (it builds `monthly_split` and `cv_folds`). Phase 3 is not required.

**Expected interaction sign (stated in advance, checked in Phase 5):** negative.

## 1. Confirm the kernel

In [ ]:
import sys
import numpy as np
import pandas as pd
print(sys.executable)

## 2. Connect to Postgres

Same convention as notebooks 02 and 03.

In [ ]:
import os
from pathlib import Path

from sqlalchemy import create_engine, text

try:
    from housing_risk.config import get_engine
    engine = get_engine()
except ImportError:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
    engine = create_engine(
        "postgresql+psycopg2://{u}:{p}@{h}:{port}/{d}".format(
            u=os.environ["PGUSER"], p=os.environ["PGPASSWORD"],
            h=os.environ.get("PGHOST", "localhost"),
            port=os.environ.get("PGPORT", "5432"),
            d=os.environ.get("PGDATABASE", "housing_risk_team"),
        )
    )

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database(), version();")).fetchone())
    for name in ("monthly_split", "cv_folds"):
        assert conn.execute(text("SELECT to_regclass(:n)"), {"n": name}).scalar(), (
            f"STOP: {name} not found -- run sql/003_phase2_split.sql first."
        )
print("monthly_split and cv_folds found.")

## 3. Run `005_phase4_features.sql`, then `003b_cv_fold_membership.sql`

`005` rebuilds `quarterly_modeling_table` and drops the `cv_fold_rows` view with it, so `003b` is run again straight afterwards. Both files can be re-run safely.

In [ ]:
def find_sql(name):
    for base in (Path("sql"), Path("..") / "sql"):
        if (base / name).exists():
            return base / name
    raise FileNotFoundError(f"{name} not found in ./sql or ../sql")


def run_sql_file(name):
    path = find_sql(name)
    raw = engine.raw_connection()
    try:
        cur = raw.cursor()
        cur.execute(open(path).read())
        raw.commit()
    except Exception:
        raw.rollback()
        raise
    finally:
        raw.close()
    print(f"ran {path}")

run_sql_file("005_phase4_features.sql")
run_sql_file("003b_cv_fold_membership.sql")

## 4. Row counts per split

Compared with the two reference points: the Phase 0 provenance figures (they count label-quarter rows before the feature history is required, so ours are lower) and Jolenn's Train feature matrix (164,460 rows, 3,009 counties, 4.85% positive).

In [ ]:
counts = pd.read_sql(
    """
    SELECT split, COUNT(1) AS n_rows, COUNT(DISTINCT county_fips) AS n_counties,
           ROUND(100.0 * AVG(decline_label), 2) AS positive_rate_pct
    FROM quarterly_modeling_table GROUP BY split ORDER BY split
    """, engine)
print(counts.to_string(index=False))
print()
train = counts.set_index("split").loc["train"]
print(f"SQL Train:                     {int(train.n_rows):,} rows, {int(train.n_counties):,} counties, {train.positive_rate_pct}% positive")
print("Jolenn's Train feature matrix: 164,460 rows, 3,009 counties, 4.85% positive")
print("Phase 0 documented (labels):   Train 173,824 / Embargo 3,021 / Test 52,087")

## 5. Table checks

No missing features, one row per county and quarter, features measured at a quarter end, and the interaction equal to the plain product.

In [ ]:
feat_cols = ["momentum_1m", "momentum_3m", "momentum_6m", "mortgage_rate_level", "mortgage_rate_roc", "rate_roc_x_momentum"]
mt = pd.read_sql("SELECT county_fips, quarter, feature_quarter_end, split, decline_label, " + ", ".join(feat_cols) +
                 " FROM quarterly_modeling_table", engine)
mt["quarter"] = pd.to_datetime(mt["quarter"]); mt["feature_quarter_end"] = pd.to_datetime(mt["feature_quarter_end"])

assert mt[feat_cols].isna().sum().sum() == 0, "STOP: missing feature values in quarterly_modeling_table"
assert not mt.duplicated(["county_fips", "quarter"]).any(), "STOP: more than one row per county and quarter"
assert mt["feature_quarter_end"].dt.month.isin([3, 6, 9, 12]).all(), "STOP: features not measured at a quarter end"
assert ((mt["quarter"] - mt["feature_quarter_end"]).dt.days == 1).all(), "STOP: features are not from the preceding quarter"
prod = mt["mortgage_rate_roc"] * mt["momentum_3m"]
assert (mt["rate_roc_x_momentum"] - prod).abs().max() < 1e-9, "STOP: interaction is not mortgage_rate_roc x momentum_3m"
print(f"{len(mt):,} rows: no missing features, one row per county and quarter, features from the preceding quarter end, "
      "interaction = rate change x 3-month momentum.")

## 6. Feature summary on the Train split

Saved to `../outputs/phase4_feature_summary.csv`.

In [ ]:
summary = mt.loc[mt["split"] == "train", feat_cols].describe().T[["mean", "std", "min", "50%", "max"]]
summary.loc["decline_label (share)"] = [mt.loc[mt["split"] == "train", "decline_label"].mean(), np.nan, 0, np.nan, 1]
os.makedirs("../outputs", exist_ok=True)
summary.to_csv("../outputs/phase4_feature_summary.csv")
summary.round(4)

## 7. Test-set dating convention

Our table is dated by **label** quarter: the Test split holds labels from 2022-Q1 to 2026-Q1 (17 quarters). Jolenn's matrix is dated by **feature** quarter: a row belongs to Test if its features fall inside the test window, and its last feature quarter has no label inside the window. That leaves 16 label quarters, starting at 2022-Q2, because the 2022-Q1 label takes its features from the embargo quarter (2021-Q4). Applying her convention to our table should give about her 48,960 rows.

In [ ]:
test = mt[mt["split"] == "test"]
n_label_dating = len(test)
n_feature_dating = int(((test["feature_quarter_end"] >= "2022-01-01") & (test["feature_quarter_end"] <= "2026-03-31")).sum())
print(f"Test rows, dated by label quarter (our convention):   {n_label_dating:,}  ({test['quarter'].nunique()} label quarters)")
print(f"Same rows, dated by feature quarter (Jolenn's):       {n_feature_dating:,}  "
      f"({test.loc[test['feature_quarter_end'] >= '2022-01-01', 'quarter'].nunique()} label quarters)")
print("Jolenn's Test feature matrix:                          48,960 rows")
print(f"Our table under her convention minus her matrix:       {n_feature_dating - 48960:+,} rows (near zero means the dating convention explains the gap)")

## 8. Row-by-row comparison with Jolenn's feature matrix (optional)

Needs her `train_features.csv` (from her `data/processed/`, which git ignores). Ask her to send it, and save it as `data/processed/jolenn_train_features.csv` on this machine. The two tables are joined on county and feature quarter end. Rows present in both should have identical feature values and labels; our table is expected to hold a few extra rows, because it uses each county's last month *with a value* in a quarter, while the pandas matrix only uses the quarter-end month and drops the row when that month is missing.

In [ ]:
JOLENN_CSV = Path("../data/processed/jolenn_train_features.csv")
if not JOLENN_CSV.exists():
    print(f"{JOLENN_CSV} not found. Ask Jolenn for her data/processed/train_features.csv and save it there to run this comparison.")
else:
    jl = pd.read_csv(JOLENN_CSV, dtype={"county_fips": str}, parse_dates=["month"])
    ours = mt[mt["split"] == "train"].rename(columns={"feature_quarter_end": "month"})
    m = ours.merge(jl, on=["county_fips", "month"], how="outer", suffixes=("_sql", "_pd"), indicator=True)
    both = m[m["_merge"] == "both"]
    only_sql, only_pd = (m["_merge"] == "left_only").sum(), (m["_merge"] == "right_only").sum()
    print(f"Rows in both: {len(both):,} | only in our SQL table: {only_sql:,} | only in Jolenn's matrix: {only_pd:,}")
    print(f"Our extra rows as a share of our Train rows: {100 * only_sql / len(ours):.2f}%")
    diffs = {f: float((both[f + '_sql'] - both[f + '_pd']).abs().max()) for f in feat_cols}
    print("Largest absolute difference on matched rows, by feature:")
    for f, d in diffs.items():
        print(f"  {f:22s} {d:.2e}")
    print(f"Label mismatches on matched rows: {int((both['decline_label_sql'] != both['decline_label_pd']).sum()):,}")

In [ ]:
# Profile of the rows our table has and Jolenn's matrix does not
if JOLENN_CSV.exists():
    extras = m[m["_merge"] == "left_only"].copy()
    extras["year"] = extras["month"].dt.year
    print(f"Extra rows: {len(extras):,} in {extras['county_fips'].nunique():,} counties; "
          f"decline share {100 * extras['decline_label_sql'].mean():.1f}% (all Train rows: {100 * ours['decline_label'].mean():.2f}%)")
    print(extras.groupby("year").agg(rows=("county_fips", "size"), declines=("decline_label_sql", "sum")).T.to_string())

## 9. Findings

- **The merged table is clean.** 226,051 rows across the four splits, no missing features, one row per county and quarter, and features measured at the preceding quarter end. The interaction is the plain product of the rate change and the 3-month momentum.
- **Train matches Jolenn's matrix closely.** 164,908 rows, 3,009 counties and 4.91% positive, against her 164,460 rows, 3,009 counties and 4.85%. Both cover the same counties, so there is no county filter on either side.
- **Every one of Jolenn's 164,460 rows is in our table, with identical values.** The largest feature difference is below 1e-15 (floating-point noise) and there are no label mismatches. Our table has 448 extra rows (0.27% of Train) and hers has none that ours lacks. The extras are expected to be quarters where the quarter-end month is missing but an earlier month in the quarter has a value; the profile above shows their years and decline share.
- **The test-set gap is fully explained by the dating convention.** Our Test split has 51,982 rows (17 label quarters). Dated by feature quarter, as Jolenn's matrix is, the same rows number 48,962 (16 label quarters), against her 48,960.
- **The new missing-value rule matters.** The earlier SQL kept rows with an empty 1-month or 3-month momentum; dropping them takes Train from 165,269 to 164,908 rows, so no model input holds an empty value.
- **Open for Phase 5:** the expected negative sign of the interaction term, and whether 1-month or 6-month momentum interacts better than 3-month.